# P4a — Chuyển đổi TFLite cho 5 mạng con của mô hình lai (Bảng 3.10) và đóng gói cho thiết bị

Chạy trên Kaggle (CPU là đủ) **sau notebook 01**. Đầu vào là thư mục kết quả của notebook 01:
5 tệp `best_exp099_*.keras`, `thesis_chosen_config.json`, `exp099_submodel_probs.npz`.

Các bước:
1. Dựng lại 5 mạng con ở **float32** rồi nạp trọng số (mô hình được huấn luyện bằng mixed precision).
2. Kiểm tra: dự đoán Keras trên S13–S17 phải trùng xác suất đã lưu ở notebook 01.
3. Chuyển 4 biến thể: Float32, Float16, dynamic range, Int8 toàn phần (dữ liệu đại diện: 500 cửa sổ huấn luyện, có cả Fall).
   Biến thể nào không chuyển được thì ghi lý do — theo cô, đó cũng là một kết quả.
4. Đánh giá từng biến thể trên S13–S17 với **cùng** TTA, λ, α, τ của EXP_099: Accuracy, Recall, F1, F2, số cửa sổ đổi nhãn so với Keras.
5. Đo độ trễ trên CPU của máy này (chỉ để tham khảo; số chính thức đo trên thiết bị ở notebook 04).
6. Đóng gói `edge_package.zip` để chép sang thiết bị.


In [46]:
%%writefile features_np.py
"""
features_np.py — Bản NumPy của các hàm tiền xử lý đặc trưng trong KLTN_EXP_099.

Dùng chung cho:
  * 03_P4_TFLite_convert.ipynb (kiểm tra TFLite trên Kaggle/PC)
  * realtime_pipeline.py và benchmark_tflite.py (chạy trên thiết bị, không cần TensorFlow đầy đủ)

Các hàm ở đây sao chép ĐÚNG logic của:
  * _mirror_skeleton_gpu()  (cell 4 của notebook EXP_099)
  * _build_features_gpu()   (cell 5 của notebook EXP_099)
Notebook 03 có ô kiểm tra so sánh với bản TensorFlow (sai lệch phải < 1e-4).
"""
import numpy as np

SEQ_LEN = 30
NUM_KP = 17
NUM_FEATURES = 51

# Thứ tự 17 khớp trong dữ liệu (đúng theo mã nguồn):
# 0 mũi | 1/2 vai T/P | 3/4 khuỷu T/P | 5/6 cổ tay T/P | 7/8 hông T/P
# 9/10 gối T/P | 11/12 cổ chân T/P | 13/14 gót chân T/P | 15/16 mũi bàn chân T/P
# Chỉ số tương ứng trong 33 điểm của MediaPipe Pose:
MEDIAPIPE_INDICES = [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
LR_SWAP = np.array([0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15], dtype=np.int64)


def mirror_skeleton_sequence(X):
    """Phản chiếu trái/phải quanh trục dọc qua tâm hông. X: (N, 30, 51) -> (N, 30, 51)."""
    X = np.asarray(X, dtype=np.float32)
    sk = X.reshape(-1, SEQ_LEN, NUM_KP, 3)
    hip_cx = (sk[:, :, 7, 0] + sk[:, :, 8, 0]) / 2.0
    mx = 2.0 * hip_cx[:, :, None] - sk[:, :, :, 0]
    sk = np.concatenate([mx[..., None], sk[:, :, :, 1:3]], axis=-1)
    sk = sk[:, :, LR_SWAP, :]
    return sk.reshape(-1, SEQ_LEN, NUM_FEATURES).astype(np.float32)


def _angle_3pts(a, b, c, eps=1e-6):
    ba = a - b
    bc = c - b
    dot = np.sum(ba * bc, axis=-1)
    den = np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1)
    cosv = np.clip(dot / np.maximum(den, eps), -1.0, 1.0)
    return np.arccos(cosv)


def _first_difference(v):
    return np.concatenate([np.zeros_like(v[:, :1]), v[:, 1:] - v[:, :-1]], axis=1)


def build_features(X):
    """
    X: (N, 30, 51) đã chuẩn hóa theo (1.7).
    Trả về (pose (N,30,51), motion (N,30,68), geometry (N,30,21), angles (N,30,8)), float32.
    """
    X = np.asarray(X, dtype=np.float32)
    n = X.shape[0]
    sk = X.reshape(n, SEQ_LEN, NUM_KP, 3)
    xy = sk[..., :2]
    pose = X

    vel = np.concatenate([np.zeros_like(xy[:, :1]), xy[:, 1:] - xy[:, :-1]], axis=1)
    acc = np.concatenate([np.zeros_like(vel[:, :1]), vel[:, 1:] - vel[:, :-1]], axis=1)
    motion = np.concatenate([vel.reshape(n, SEQ_LEN, 34), acc.reshape(n, SEQ_LEN, 34)], axis=-1)

    l_sh, r_sh = xy[:, :, 1], xy[:, :, 2]
    l_hip, r_hip = xy[:, :, 7], xy[:, :, 8]
    l_knee, r_knee = xy[:, :, 9], xy[:, :, 10]
    l_ank, r_ank = xy[:, :, 11], xy[:, :, 12]
    sh_mid = (l_sh + r_sh) / 2.0
    hip_mid = (l_hip + r_hip) / 2.0

    x_all, y_all = xy[..., 0], xy[..., 1]
    body_width = x_all.max(axis=2) - x_all.min(axis=2)
    body_height = y_all.max(axis=2) - y_all.min(axis=2)
    aspect = body_width / np.maximum(body_height, 1e-6)
    shoulder_width = np.linalg.norm(l_sh - r_sh, axis=-1)
    hip_width = np.linalg.norm(l_hip - r_hip, axis=-1)
    torso_vec = sh_mid - hip_mid
    torso_len = np.linalg.norm(torso_vec, axis=-1)
    torso_angle = np.arctan2(np.abs(torso_vec[..., 0]), np.abs(torso_vec[..., 1]) + 1e-6)  # radian
    left_knee = _angle_3pts(l_hip, l_knee, l_ank)
    right_knee = _angle_3pts(r_hip, r_knee, r_ank)
    geometry_base = np.stack([body_width, body_height, aspect, shoulder_width, hip_width,
                              torso_len, torso_angle, left_knee, right_knee], axis=-1)

    speed = np.linalg.norm(vel, axis=-1)
    accel_mag = np.linalg.norm(acc, axis=-1)
    mean_speed = speed.mean(axis=2)
    max_speed = speed.max(axis=2)
    mean_accel = accel_mag.mean(axis=2)
    max_accel = accel_mag.max(axis=2)
    lower_extension = (np.linalg.norm(l_hip - l_ank, axis=-1) + np.linalg.norm(r_hip - r_ank, axis=-1)) / 2.0
    d_torso_angle = _first_difference(torso_angle)
    transition = np.stack([
        d_torso_angle, np.abs(d_torso_angle), _first_difference(aspect),
        _first_difference(body_height), lower_extension, _first_difference(lower_extension),
        mean_speed, max_speed, mean_accel, max_accel,
        _first_difference(mean_speed), _first_difference(torso_len),
    ], axis=-1)
    geometry = np.concatenate([geometry_base, transition], axis=-1)

    angles = np.stack([
        np.cos(_angle_3pts(l_sh, xy[:, :, 3], xy[:, :, 5])),
        np.cos(_angle_3pts(r_sh, xy[:, :, 4], xy[:, :, 6])),
        np.cos(_angle_3pts(xy[:, :, 3], l_sh, l_hip)),
        np.cos(_angle_3pts(xy[:, :, 4], r_sh, r_hip)),
        np.cos(_angle_3pts(l_sh, l_hip, l_knee)),
        np.cos(_angle_3pts(r_sh, r_hip, r_knee)),
        np.cos(_angle_3pts(l_hip, l_knee, l_ank)),
        np.cos(_angle_3pts(r_hip, r_knee, r_ank)),
    ], axis=-1)

    f32 = lambda a: np.asarray(a, dtype=np.float32)
    return f32(pose), f32(motion), f32(geometry), f32(angles)


def torso_angle_deg_last(X, n_theta=5):
    """θ̄_j (độ): trung bình góc nghiêng thân (2.6) trên n_theta khung cuối của mỗi cửa sổ."""
    sk = np.asarray(X, dtype=np.float32).reshape(-1, SEQ_LEN, NUM_KP, 3)
    sh_mid = (sk[:, :, 1, :2] + sk[:, :, 2, :2]) / 2.0
    hip_mid = (sk[:, :, 7, :2] + sk[:, :, 8, :2]) / 2.0
    v = sh_mid - hip_mid
    theta = np.degrees(np.arctan2(np.abs(v[..., 0]), np.abs(v[..., 1]) + 1e-6))
    return theta[:, -n_theta:].mean(axis=1)


# ---------------------------------------------------------------------
# Kết hợp dự đoán của mô hình lai: TTA (2.7a), trung bình λ (2.7), hợp nhất logit (2.8)
# ---------------------------------------------------------------------
def logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


def tta_blend(raw_prob, mirror_prob, beta, mode):
    beta = float(beta)
    raw_prob = np.asarray(raw_prob, dtype=np.float64)
    if mode == 'raw' or beta <= 0.0:
        return raw_prob
    mirror_prob = np.asarray(mirror_prob, dtype=np.float64)
    if mode == 'prob':
        return (1.0 - beta) * raw_prob + beta * mirror_prob
    if mode == 'logit':
        return sigmoid((1.0 - beta) * logit(raw_prob) + beta * logit(mirror_prob))
    raise ValueError(mode)


def fuse_logit(p_tf, p_pmg, alpha):
    return sigmoid((1.0 - alpha) * logit(p_tf) + alpha * logit(p_pmg))


Overwriting features_np.py


In [47]:
%%writefile alert_manager.py
"""
alert_manager.py — Thuật toán 2.2: Bộ quản lý cảnh báo bốn trạng thái.

Dùng CHUNG một mã cho:
  * đánh giá phát lại trên UP-Fall (Thuật toán 3.1, notebook 02)
  * hệ thống chạy thời gian thực trên thiết bị (realtime_pipeline.py)

Trạng thái: NORMAL (BÌNH_THƯỜNG) -> SUSPECT (NGHI_NGỜ) -> CONFIRMED (XÁC_NHẬN) -> COOLDOWN (CHỜ)
"""
from dataclasses import dataclass, field

NORMAL, SUSPECT, CONFIRMED, COOLDOWN = 'NORMAL', 'SUSPECT', 'CONFIRMED', 'COOLDOWN'


@dataclass
class AlertParams:
    tau: float                 # ngưỡng phân loại τ
    theta_lie: float = 60.0    # ngưỡng góc nằm θ_lie (độ); 0 = bỏ bước xác nhận tư thế
    t_post: float = 3.0        # T_post (giây): thời gian tối đa chờ xác nhận
    t_cool: float = 30.0       # T_c (giây): thời gian chờ sau mỗi cảnh báo


@dataclass
class AlertManager:
    params: AlertParams
    state: str = NORMAL
    t_s: float = 0.0           # thời điểm bắt đầu nghi ngờ
    p_s: float = 0.0           # xác suất lúc bắt đầu nghi ngờ
    t_alert: float = float('-inf')
    alerts: list = field(default_factory=list)   # danh sách (t_alert, p_s)

    def reset(self):
        self.state = NORMAL
        self.t_s = 0.0
        self.p_s = 0.0
        self.t_alert = float('-inf')
        self.alerts = []

    def update(self, t_j, p_j, theta_j):
        """
        Xử lý một cửa sổ mới (bước 2 của Thuật toán 2.2).
        t_j: thời điểm kết thúc cửa sổ (giây); p_j: xác suất Fall; theta_j: θ̄_j (độ).
        Trả về True nếu phát cảnh báo tại cửa sổ này.
        """
        P = self.params
        # 2.1
        if self.state == COOLDOWN and (t_j - self.t_alert) >= P.t_cool:
            self.state = NORMAL
        # 2.2
        if self.state == NORMAL and p_j >= P.tau:
            self.state = SUSPECT
            self.t_s = t_j
            self.p_s = p_j
        # 2.3
        if self.state == SUSPECT:
            if theta_j >= P.theta_lie:
                self.state = CONFIRMED
            elif (t_j - self.t_s) > P.t_post:
                self.state = NORMAL
        # 2.4
        if self.state == CONFIRMED:
            self.t_alert = t_j
            self.alerts.append((t_j, self.p_s))
            self.state = COOLDOWN
            return True
        return False


def run_alert_manager(times, probs, thetas, params):
    """Chạy Thuật toán 2.2 trên một dãy cửa sổ đã sắp theo thời gian; trả về danh sách cảnh báo."""
    am = AlertManager(params)
    for t, p, th in zip(times, probs, thetas):
        am.update(float(t), float(p), float(th))
    return am.alerts


def run_every_window(times, probs, tau):
    """Chính sách so sánh: mỗi cửa sổ có p >= τ là một cảnh báo."""
    return [(float(t), float(p)) for t, p in zip(times, probs) if p >= tau]


Overwriting alert_manager.py


In [48]:
%%writefile hybrid_tflite.py
"""
hybrid_tflite.py — Suy luận mô hình lai PMG–Transformer (Thuật toán 2.1) bằng TFLite.

Thư mục mô hình (xuất từ notebook 03) gồm:
    hybrid_config.json
    <variant>/transformer_42.tflite, transformer_777.tflite,
              pmg_42.tflite, pmg_123.tflite, pmg_2026.tflite
"""
import json
import time
from pathlib import Path

import numpy as np

import features_np as F


def _make_interpreter(path, num_threads):
    try:
        from ai_edge_litert.interpreter import Interpreter          # gói mới (LiteRT)
    except ImportError:
        try:
            from tflite_runtime.interpreter import Interpreter      # gói cũ
        except ImportError:
            import tensorflow as tf                                 # đủ TensorFlow
            Interpreter = tf.lite.Interpreter
    it = Interpreter(model_path=str(path), num_threads=num_threads)
    it.allocate_tensors()
    return it


class _Sub:
    """Một mạng con TFLite; ánh xạ tên đầu vào (pose/motion/geometry/angles/skeleton_sequence)."""

    def __init__(self, path, num_threads):
        self.it = _make_interpreter(path, num_threads)
        self.inputs = []
        for d in self.it.get_input_details():
            name = d['name'].replace('serving_default_', '').split(':')[0]
            self.inputs.append((name, d['index']))
        self.out = self.it.get_output_details()[0]['index']

    def __call__(self, feeds):
        for name, idx in self.inputs:
            self.it.set_tensor(idx, feeds[name])
        self.it.invoke()
        return float(self.it.get_tensor(self.out).reshape(-1)[0])


class HybridFallModel:
    def __init__(self, model_dir, variant='float32', num_threads=4):
        model_dir = Path(model_dir)
        self.cfg = json.load(open(model_dir / 'hybrid_config.json'))
        vdir = model_dir / variant
        self.alpha = float(self.cfg['alpha_pmg'])
        self.tau = float(self.cfg['threshold_tau'])
        self.lam_tf = np.asarray(self.cfg['lambda_transformer'], float)
        self.lam_pmg = np.asarray(self.cfg['lambda_pmg'], float)
        self.tf_seeds = self.cfg['transformer_seeds']
        self.pmg_seeds = self.cfg['pmg_seeds']
        self.tta_tf = self.cfg['tta_transformer']
        self.tta_pmg = self.cfg['tta_pmg']
        self.tf_models = [_Sub(vdir / f'transformer_{s}.tflite', num_threads) for s in self.tf_seeds]
        self.pmg_models = [_Sub(vdir / f'pmg_{s}.tflite', num_threads) for s in self.pmg_seeds]
        self.last_timing = {}

    @staticmethod
    def _need_mirror(choice):
        return choice['mode'] != 'raw' and float(choice['beta']) > 0.0

    def predict(self, window):
        """window: (30, 51) float32 đã chuẩn hóa theo (1.7). Trả về p (xác suất Fall)."""
        t0 = time.perf_counter()
        X = np.asarray(window, np.float32).reshape(1, F.SEQ_LEN, F.NUM_FEATURES)
        need_m = any(self._need_mirror(self.tta_tf[str(s)]) for s in self.tf_seeds) or \
                 any(self._need_mirror(self.tta_pmg[str(s)]) for s in self.pmg_seeds)
        Xm = F.mirror_skeleton_sequence(X) if need_m else None
        P, M, G, A = F.build_features(X)
        feeds = {'skeleton_sequence': X, 'pose': P, 'motion': M, 'geometry': G, 'angles': A}
        if need_m:
            Pm, Mm, Gm, Am = F.build_features(Xm)
            feeds_m = {'skeleton_sequence': Xm, 'pose': Pm, 'motion': Mm, 'geometry': Gm, 'angles': Am}
        t1 = time.perf_counter()

        p_tf_list = []
        for s, m in zip(self.tf_seeds, self.tf_models):
            c = self.tta_tf[str(s)]
            raw = m(feeds)
            mir = m(feeds_m) if self._need_mirror(c) else raw
            p_tf_list.append(F.tta_blend(raw, mir, c['beta'], c['mode']))
        t2 = time.perf_counter()

        p_pmg_list = []
        for s, m in zip(self.pmg_seeds, self.pmg_models):
            c = self.tta_pmg[str(s)]
            raw = m(feeds)
            mir = m(feeds_m) if self._need_mirror(c) else raw
            p_pmg_list.append(F.tta_blend(raw, mir, c['beta'], c['mode']))
        t3 = time.perf_counter()

        p_tf = float(np.average(np.asarray(p_tf_list, float).ravel(), weights=self.lam_tf))
        p_pmg = float(np.average(np.asarray(p_pmg_list, float).ravel(), weights=self.lam_pmg))
        p = float(F.fuse_logit(p_tf, p_pmg, self.alpha))
        t4 = time.perf_counter()
        self.last_timing = {
            'features_ms': 1e3 * (t1 - t0), 'transformer_branch_ms': 1e3 * (t2 - t1),
            'pmg_branch_ms': 1e3 * (t3 - t2), 'fusion_ms': 1e3 * (t4 - t3), 'total_ms': 1e3 * (t4 - t0),
        }
        return p


Overwriting hybrid_tflite.py


In [49]:
# ============================================================
# 1. CẤU HÌNH
# ============================================================
import os, json, time, shutil, platform
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
from pathlib import Path
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.metrics import accuracy_score, recall_score, f1_score, fbeta_score, precision_score
mixed_precision.set_global_policy('float32')     # dựng lại ở float32 để chuyển TFLite
import features_np as F

EXP_DIR = Path('/kaggle/working/KLTN_EXP_099')            # thư mục kết quả notebook 01 (sửa nếu đã tải lên dạng dataset)
NPZ_DIR = Path('/kaggle/input/datasets/anhoa1/upfall-camera2-processing')
OUT = Path('/kaggle/working/P4_tflite'); OUT.mkdir(parents=True, exist_ok=True)
N_REPRESENTATIVE = 500
print('TensorFlow', tf.__version__, '| CPU:', platform.processor() or platform.machine())


TensorFlow 2.20.0 | CPU: x86_64


In [50]:
# ============================================================
# 2. KIẾN TRÚC — chép nguyên từ KLTN_EXP_099 (cell 5 và cell 6), không thay đổi
# ============================================================
SEQ_LEN = 30; NUM_KP = 17; NUM_FEATURES = 51
D_MODEL = 64; NUM_HEADS = 4; FF_DIM = 128; NUM_BLOCKS = 3; DROPOUT = 0.20
PMG_D_MODEL = 96; PMG_FF = 192; PMG_BLOCKS = 2
TCN_CHANNELS = [64, 96, 96]; TCN_DILATIONS = [1, 2, 4]
GEOMETRY_FEATURE_DIM = 21; ANGLE_FEATURE_DIM = 8; MOTION_ANGLE_INTERACTION_DIM = 32
TRANSFORMER_LR = 5e-4; PMG_LR = 3e-4; STEPS_PER_EXECUTION = 1

# ============================================================
# 5. CUSTOM LAYERS
# ============================================================

@keras.utils.register_keras_serializable(package='UPFallV7')
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, d_model, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = int(sequence_length)
        self.d_model = int(d_model)
        self.position_embeddings = layers.Embedding(
            input_dim=self.sequence_length,
            output_dim=self.d_model
        )

    def build(self, input_shape):
        self.position_embeddings.build((None,))
        super().build(input_shape)

    def call(self, x):
        positions = tf.range(self.sequence_length)
        return x + self.position_embeddings(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'sequence_length': self.sequence_length,
            'd_model': self.d_model,
        })
        return cfg


@keras.utils.register_keras_serializable(package='UPFallV7')
class AttentionPool1D(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score = layers.Dense(1)

    def build(self, input_shape):
        self.score.build(input_shape)
        super().build(input_shape)

    def call(self, x):
        a = tf.nn.softmax(self.score(x), axis=1)
        return tf.reduce_sum(a*x, axis=1)


# ============================================================
# 6. MODEL BUILDERS
# ============================================================

def transformer_block(x, d_model, num_heads, ff_dim, dropout):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout
    )(x, x)

    x = layers.LayerNormalization(epsilon=1e-6)(
        x + layers.Dropout(dropout)(attn)
    )

    ff = layers.Dense(ff_dim, activation='gelu')(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)

    return layers.LayerNormalization(epsilon=1e-6)(
        x + layers.Dropout(dropout)(ff)
    )


def build_precision_transformer(seed):
    tf.keras.utils.set_random_seed(seed)

    inputs = keras.Input((30,51), name='skeleton_sequence')
    x = layers.LayerNormalization()(inputs)
    x = layers.Dense(D_MODEL, name='feature_projection')(x)
    x = PositionalEmbedding(30, D_MODEL, name='positional_embedding')(x)

    for _ in range(NUM_BLOCKS):
        x = transformer_block(x, D_MODEL, NUM_HEADS, FF_DIM, DROPOUT)

    # Mix average and max pooling to preserve discriminative peaks.
    avg = layers.GlobalAveragePooling1D()(x)
    mx = layers.GlobalMaxPooling1D()(x)
    x = layers.Concatenate()([avg, mx])

    x = layers.Dense(64, activation='gelu')(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(32, activation='gelu')(x)
    x = layers.Dropout(0.20)(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='fall_probability')(x)

    model = keras.Model(inputs, out, name=f'precision_transformer_{seed}')
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=TRANSFORMER_LR,
            weight_decay=1e-4
        ),
        loss='binary_crossentropy',
        steps_per_execution=STEPS_PER_EXECUTION,
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='roc_auc'),
            keras.metrics.AUC(curve='PR', name='pr_auc'),
        ]
    )
    return model


def tcn_block(x, filters, dilation):
    residual = x
    y = layers.Conv1D(
        filters, 3, padding='same', dilation_rate=dilation,
        activation='gelu'
    )(x)
    y = layers.BatchNormalization()(y)
    y = layers.Dropout(0.20)(y)

    y = layers.Conv1D(
        filters, 3, padding='same', dilation_rate=dilation,
        activation='gelu'
    )(y)
    y = layers.BatchNormalization()(y)

    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1, padding='same')(residual)

    return layers.LayerNormalization()(residual + y)


def build_pmg(seed):
    tf.keras.utils.set_random_seed(seed)

    pose_in = keras.Input((30,51), name='pose')
    p = layers.LayerNormalization()(pose_in)
    p = layers.Dense(PMG_D_MODEL, activation='gelu')(p)
    p = PositionalEmbedding(30, PMG_D_MODEL)(p)
    for _ in range(PMG_BLOCKS):
        p = transformer_block(p, PMG_D_MODEL, 4, PMG_FF, 0.20)
    p = AttentionPool1D(name='pose_pool')(p)

    motion_in = keras.Input((30,68), name='motion')
    m = layers.LayerNormalization()(motion_in)
    for ch, dilation in zip(TCN_CHANNELS, TCN_DILATIONS):
        m = tcn_block(m, ch, dilation)
    m = AttentionPool1D(name='motion_pool')(m)

    geo_in = keras.Input((30,GEOMETRY_FEATURE_DIM), name='geometry')
    angle_in = keras.Input((30,ANGLE_FEATURE_DIM), name='angles')
    g = layers.LayerNormalization()(geo_in)
    g = layers.Conv1D(32, 3, padding='same', activation='gelu')(g)
    g = layers.Conv1D(48, 3, padding='same', activation='gelu')(g)
    g = AttentionPool1D(name='geometry_pool')(g)

    # EXP_099 controlled change: compact angle temporal branch.
    a = layers.LayerNormalization(name='angle_norm')(angle_in)
    a = layers.Conv1D(32, 3, padding='same', activation='gelu', name='angle_conv1')(a)
    a = layers.Conv1D(32, 3, padding='same', activation='gelu', name='angle_conv2')(a)
    a = AttentionPool1D(name='angle_pool')(a)
    # EXP_099 controlled change: normalize pooled angle embedding before fusion.
    a = layers.LayerNormalization(
        epsilon=1e-6,
        name='angle_post_pool_norm'
    )(a)

    # EXP_099 controlled change:
    # compact interaction between pooled motion and pooled angle embeddings.
    ma_m = layers.Dense(
        MOTION_ANGLE_INTERACTION_DIM,
        activation='gelu',
        name='motion_angle_motion_projection'
    )(m)
    ma_a = layers.Dense(
        MOTION_ANGLE_INTERACTION_DIM,
        activation='gelu',
        name='motion_angle_angle_projection'
    )(a)
    ma = layers.Multiply(
        name='motion_angle_interaction_product'
    )([ma_m, ma_a])
    ma = layers.LayerNormalization(
        epsilon=1e-6,
        name='motion_angle_interaction_norm'
    )(ma)

    x = layers.Concatenate(
        name='pmg_specialist_fusion'
    )([p,m,g,a,ma])
    x = layers.Dense(192, activation='gelu')(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(96, activation='gelu')(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(48, activation='gelu')(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='fall_probability')(x)

    model = keras.Model([pose_in,motion_in,geo_in,angle_in], out, name=f'pmg_{seed}')
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=PMG_LR,
            weight_decay=1e-4
        ),
        loss='binary_crossentropy',
        steps_per_execution=STEPS_PER_EXECUTION,
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='roc_auc'),
            keras.metrics.AUC(curve='PR', name='pr_auc'),
        ]
    )
    return model


In [51]:
# ============================================================
# 3. NẠP 5 MẠNG CON
# ============================================================
cfg = json.load(open(EXP_DIR/'thesis_chosen_config.json'))
ALPHA = float(cfg['alpha_pmg']); TAU = float(cfg['threshold_tau'])
LAM_TF = np.asarray(cfg['lambda_transformer'], float); LAM_PMG = np.asarray(cfg['lambda_pmg'], float)
TF_SEEDS = cfg['transformer_seeds']; PMG_SEEDS = cfg['pmg_seeds']
TTA_TF = cfg['tta_transformer']; TTA_PMG = cfg['tta_pmg']
print('α =', ALPHA, '| τ =', TAU, '| λ_TF =', LAM_TF, '| λ_PMG =', LAM_PMG)
print('TTA Transformer:', TTA_TF); print('TTA PMG:', TTA_PMG)

models = {}
for s in TF_SEEDS:
    m = build_precision_transformer(int(s)); m.load_weights(EXP_DIR/f'best_exp099_transformer_{s}.keras')
    models[f'transformer_{s}'] = m
for s in PMG_SEEDS:
    m = build_pmg(int(s)); m.load_weights(EXP_DIR/f'best_exp099_pmg_{s}.keras')
    models[f'pmg_{s}'] = m
for k, m in models.items():
    print(f'{k:18s} tham số huấn luyện được: {sum(int(np.prod(w.shape)) for w in m.trainable_weights):,}')


α = 0.525 | τ = 0.7840000000000003 | λ_TF = [0.975 0.025] | λ_PMG = [0.9 0.1 0. ]
TTA Transformer: {'42': {'mode': 'logit', 'beta': 0.1, 'val_pr_auc': 0.8331999623659634, 'val_roc_auc': 0.9922469490308686}, '777': {'mode': 'logit', 'beta': 0.3, 'val_pr_auc': 0.7941415401549425, 'val_roc_auc': 0.9838478104809764}}
TTA PMG: {'42': {'mode': 'raw', 'beta': 0.0, 'val_pr_auc': 0.8112850123860957, 'val_roc_auc': 0.981667264895908}, '123': {'mode': 'prob', 'beta': 0.1, 'val_pr_auc': 0.7611671894011889, 'val_roc_auc': 0.9867372577171573}, '2026': {'mode': 'logit', 'beta': 0.2, 'val_pr_auc': 0.7538940130918053, 'val_roc_auc': 0.9802315147164393}}
transformer_42     tham số huấn luyện được: 116,135
transformer_777    tham số huấn luyện được: 116,135
pmg_42             tham số huấn luyện được: 394,125
pmg_123            tham số huấn luyện được: 394,125
pmg_2026           tham số huấn luyện được: 394,125


In [52]:
# ============================================================
# 4. DỮ LIỆU + KIỂM TRA KERAS (phải trùng xác suất notebook 01)
# ============================================================
dte = np.load(NPZ_DIR/'upfall_test.npz', allow_pickle=True)
dtr = np.load(NPZ_DIR/'upfall_train.npz', allow_pickle=True)
X_test = dte['X'].astype(np.float32); y_test = dte['y'].astype(int)
X_test_m = F.mirror_skeleton_sequence(X_test)
FEATS = {'raw': F.build_features(X_test), 'mirror': F.build_features(X_test_m)}
XS = {'raw': X_test, 'mirror': X_test_m}

def keras_predict(name, which):
    m = models[name]
    if name.startswith('transformer'):
        return m.predict(XS[which], batch_size=512, verbose=0).reshape(-1)
    P, M, G, A = FEATS[which]
    return m.predict([P, M, G, A], batch_size=512, verbose=0).reshape(-1)

KERAS = {(n, w): keras_predict(n, w) for n in models for w in ['raw', 'mirror']}

ref = np.load(EXP_DIR/'exp099_submodel_probs.npz')
for i, s in enumerate(TF_SEEDS):
    print(f'transformer_{s}: |Δ| raw = {np.abs(KERAS[(f"transformer_{s}","raw")] - ref["tr_raw_test"][i]).max():.2e}, '
          f'mirror = {np.abs(KERAS[(f"transformer_{s}","mirror")] - ref["tr_mirror_test"][i]).max():.2e}')
for i, s in enumerate(PMG_SEEDS):
    print(f'pmg_{s}: |Δ| raw = {np.abs(KERAS[(f"pmg_{s}","raw")] - ref["pmg_raw_test"][i]).max():.2e}, '
          f'mirror = {np.abs(KERAS[(f"pmg_{s}","mirror")] - ref["pmg_mirror_test"][i]).max():.2e}')
print('Sai lệch ~1e-3 là bình thường (huấn luyện mixed precision trên GPU, đánh giá float32 trên CPU).')

def combine(sub):   # sub[(name, which)] -> xác suất; áp TTA, λ, α đúng như EXP_099
    p_tf = np.average(np.stack([F.tta_blend(sub[(f'transformer_{s}','raw')], sub[(f'transformer_{s}','mirror')],
                                            TTA_TF[str(s)]['beta'], TTA_TF[str(s)]['mode']) for s in TF_SEEDS]),
                      axis=0, weights=LAM_TF)
    p_pmg = np.average(np.stack([F.tta_blend(sub[(f'pmg_{s}','raw')], sub[(f'pmg_{s}','mirror')],
                                             TTA_PMG[str(s)]['beta'], TTA_PMG[str(s)]['mode']) for s in PMG_SEEDS]),
                       axis=0, weights=LAM_PMG)
    return F.fuse_logit(p_tf, p_pmg, ALPHA)

p_keras = combine(KERAS)
print('|Δ| xác suất mô hình lai so với notebook 01:', float(np.abs(p_keras - ref['final_test']).max()))
pred_keras = (p_keras >= TAU).astype(int)


transformer_42: |Δ| raw = 5.36e-07, mirror = 7.15e-07
transformer_777: |Δ| raw = 8.05e-07, mirror = 1.01e-06
pmg_42: |Δ| raw = 3.37e-06, mirror = 1.37e-06
pmg_123: |Δ| raw = 8.94e-07, mirror = 8.34e-07
pmg_2026: |Δ| raw = 1.73e-06, mirror = 1.70e-06
Sai lệch ~1e-3 là bình thường (huấn luyện mixed precision trên GPU, đánh giá float32 trên CPU).
|Δ| xác suất mô hình lai so với notebook 01: 7.080340021503595e-07


In [53]:
# ============================================================
# 5. CHUYỂN ĐỔI TFLITE — 4 biến thể
# ============================================================
rng = np.random.default_rng(2026)
ytr = dtr['y'].astype(int)
idx_f = np.flatnonzero(ytr == 1); idx_n = np.flatnonzero(ytr == 0)
n_f = min(len(idx_f), N_REPRESENTATIVE // 5)
rep_idx = np.concatenate([rng.choice(idx_f, n_f, replace=False), rng.choice(idx_n, N_REPRESENTATIVE - n_f, replace=False)])
X_rep = dtr['X'][rep_idx].astype(np.float32)
P_rep, M_rep, G_rep, A_rep = F.build_features(X_rep)
REP = {'skeleton_sequence': X_rep, 'pose': P_rep, 'motion': M_rep, 'geometry': G_rep, 'angles': A_rep}
print(f'Dữ liệu đại diện cho Int8: {len(rep_idx)} cửa sổ huấn luyện ({n_f} Fall)')

VARIANTS = ['float32', 'float16', 'dynamic_range', 'int8_full']

def input_order(tfl_bytes):
    it = tf.lite.Interpreter(model_content=tfl_bytes); it.allocate_tensors()
    return [d['name'].replace('serving_default_', '').split(':')[0] for d in it.get_input_details()]

def convert(model, variant, f32_bytes=None):
    c = tf.lite.TFLiteConverter.from_keras_model(model)
    if variant == 'float16':
        c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
    elif variant == 'dynamic_range':
        c.optimizations = [tf.lite.Optimize.DEFAULT]
    elif variant == 'int8_full':
        order = input_order(f32_bytes)
        def rep():
            for i in range(len(rep_idx)):
                yield [REP[k][i:i+1] for k in order]
        c.optimizations = [tf.lite.Optimize.DEFAULT]; c.representative_dataset = rep
        c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        c.inference_input_type = tf.float32; c.inference_output_type = tf.float32
    return c.convert()

CONVERT_LOG = []
for v in VARIANTS:
    (OUT/'edge_package'/'models'/v).mkdir(parents=True, exist_ok=True)
F32 = {}
for name, m in models.items():
    for v in VARIANTS:
        t0 = time.time()
        try:
            b = convert(m, v, F32.get(name))
            if v == 'float32': F32[name] = b
            (OUT/'edge_package'/'models'/v/f'{name}.tflite').write_bytes(b)
            CONVERT_LOG.append({'model': name, 'variant': v, 'ok': True, 'size_kb': len(b)/1024,
                                'time_s': time.time()-t0, 'error': ''})
        except Exception as e:
            CONVERT_LOG.append({'model': name, 'variant': v, 'ok': False, 'size_kb': np.nan,
                                'time_s': time.time()-t0, 'error': str(e).splitlines()[0][:300]})
conv_df = pd.DataFrame(CONVERT_LOG)
display(conv_df)
conv_df.to_csv(OUT/'tflite_conversion_log.csv', index=False)


Dữ liệu đại diện cho Int8: 500 cửa sổ huấn luyện (100 Fall)
Saved artifact at '/tmp/tmpwagp56di'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30, 51), dtype=tf.float32, name='skeleton_sequence')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138863950408848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685493712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685494864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685495824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685495632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496784: TensorS

W0000 00:00:1795506409.423774      49 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1795506409.423848      49 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1795506409.464138      49 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


Saved artifact at '/tmp/tmpojdrx_5v'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30, 51), dtype=tf.float32, name='skeleton_sequence')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138863950408848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685493712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685494864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685495824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685495632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13886268549736

W0000 00:00:1795506412.380220      49 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1795506412.380274      49 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


Saved artifact at '/tmp/tmpuevv2z24'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30, 51), dtype=tf.float32, name='skeleton_sequence')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138863950408848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685493712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685494864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685495824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685495632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862685496784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13886268549736

W0000 00:00:1795506415.279617      49 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1795506415.279642      49 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


... (31 outputs left out)

W0000 00:00:1795506509.050880      49 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1795506509.050906      49 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Saved artifact at '/tmp/tmpbggubl2e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 30, 51), dtype=tf.float32, name='pose'), TensorSpec(shape=(None, 30, 68), dtype=tf.float32, name='motion'), TensorSpec(shape=(None, 30, 21), dtype=tf.float32, name='geometry'), TensorSpec(shape=(None, 30, 8), dtype=tf.float32, name='angles')]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138862686548432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862686549392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862686549008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862686550352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862686549968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862686549776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862686547280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138862686550160: TensorSpec(s

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1795506514.791526      49 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1795506514.791576      49 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32


,model,variant,ok,size_kb,time_s,error
0,transformer_42,float32,True,493.265625,3.259057,
1,transformer_42,float16,True,273.320312,2.861872,
2,transformer_42,dynamic_range,True,183.234375,2.868113,
3,transformer_42,int8_full,True,192.101562,4.878517,
4,transformer_777,float32,True,493.507812,3.613499,
5,transformer_777,float16,True,273.562500,2.882710,
6,transformer_777,dynamic_range,True,183.445312,2.874044,
7,transformer_777,int8_full,True,192.250000,4.890052,
8,pmg_42,float32,True,1607.781250,5.670425,
9,pmg_42,float16,True,850.312500,6.532584,


In [54]:
# ============================================================
# 6. ĐÁNH GIÁ TỪNG BIẾN THỂ TRÊN S13–S17 + BẢNG 3.10
# ============================================================
def tflite_predict(path, name, which, batch=256):
    it = tf.lite.Interpreter(model_path=str(path), num_threads=os.cpu_count())
    ins = it.get_input_details()
    names = [d['name'].replace('serving_default_', '').split(':')[0] for d in ins]
    if name.startswith('transformer'):
        data = {'skeleton_sequence': XS[which]}
    else:
        P, M, G, A = FEATS[which]; data = {'pose': P, 'motion': M, 'geometry': G, 'angles': A}
    n = len(y_test); out = np.zeros(n, np.float32)
    try:                                   # thử chạy theo lô cho nhanh
        for d in ins: it.resize_tensor_input(d['index'], [batch] + list(d['shape'][1:]))
        it.allocate_tensors(); bs = batch
    except Exception:
        it.allocate_tensors(); bs = 1
    od = it.get_output_details()[0]['index']
    for s0 in range(0, n, bs):
        s1 = min(n, s0+bs); k = s1 - s0
        for d, nm in zip(ins, names):
            x = data[nm][s0:s1]
            if k < bs: x = np.concatenate([x, np.repeat(x[-1:], bs-k, axis=0)])
            it.set_tensor(d['index'], x.astype(np.float32))
        it.invoke()
        out[s0:s1] = it.get_tensor(od).reshape(-1)[:k]
    return out

rows = [{
    'Phiên bản': 'Keras Float32 (mô hình lai, kết quả kiểm thử ở Bảng 3.8)',
    'Kích thước tệp (KB)': np.nan,
    'Accuracy (%)': 100*accuracy_score(y_test, pred_keras), 'Recall (%)': 100*recall_score(y_test, pred_keras),
    'F1 (%)': 100*f1_score(y_test, pred_keras), 'F2 (%)': 100*fbeta_score(y_test, pred_keras, beta=2),
    'Số cửa sổ đổi nhãn so với Keras': 0, '|Δp| lớn nhất': 0.0,
}]
VARIANT_PROBS = {}
for v in VARIANTS:
    ok = conv_df[(conv_df.variant == v)].ok.all()
    if not ok:
        rows.append({'Phiên bản': f'TFLite {v}', 'Ghi chú': 'không chuyển đổi được — xem tflite_conversion_log.csv'})
        continue
    sub = {(n, w): tflite_predict(OUT/'edge_package'/'models'/v/f'{n}.tflite', n, w)
           for n in models for w in ['raw', 'mirror']}
    p = combine(sub); pred = (p >= TAU).astype(int); VARIANT_PROBS[v] = p
    rows.append({
        'Phiên bản': f'TFLite {v}',
        'Kích thước tệp (KB)': conv_df[conv_df.variant == v].size_kb.sum(),
        'Accuracy (%)': 100*accuracy_score(y_test, pred), 'Recall (%)': 100*recall_score(y_test, pred),
        'F1 (%)': 100*f1_score(y_test, pred), 'F2 (%)': 100*fbeta_score(y_test, pred, beta=2),
        'Số cửa sổ đổi nhãn so với Keras': int((pred != pred_keras).sum()),
        '|Δp| lớn nhất': float(np.abs(p - p_keras).max()),
    })
table_3_10 = pd.DataFrame(rows)
display(table_3_10.round(3))
table_3_10.to_csv(OUT/'table_3_10_quantization_accuracy.csv', index=False)
print('Kích thước = tổng 5 mạng con. Cột độ trễ của Bảng 3.10 lấy từ notebook 04 (đo trên thiết bị).')


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use 

,Phiên bản,Kích thước tệp (KB),Accuracy (%),Recall (%),F1 (%),F2 (%),Số cửa sổ đổi nhãn so với Keras,|Δp| lớn nhất
0,"Keras Float32 (mô hình lai, kết quả kiểm thử ở...",NaN,98.767,68.0,69.744,68.687,0,0.000
1,TFLite float32,5811.797,98.767,68.0,69.744,68.687,0,0.000
2,TFLite float16,3099.500,98.767,68.0,69.744,68.687,0,0.001
3,TFLite dynamic_range,1904.414,98.788,69.0,70.408,69.556,1,0.035
4,TFLite int8_full,1980.555,98.621,48.0,59.259,51.948,41,0.851


Kích thước = tổng 5 mạng con. Cột độ trễ của Bảng 3.10 lấy từ notebook 04 (đo trên thiết bị).


In [55]:
# ============================================================
# 7. ĐỘ TRỄ TRÊN CPU MÁY NÀY (tham khảo) — cùng mã suy luận sẽ chạy trên thiết bị
# ============================================================
from hybrid_tflite import HybridFallModel

pkg = OUT/'edge_package'
hybrid_cfg = {**cfg, 'variants': [v for v in VARIANTS if conv_df[conv_df.variant == v].ok.all()],
              'mediapipe_indices': F.MEDIAPIPE_INDICES, 'seq_len': 30, 'stride': 15}
json.dump(hybrid_cfg, open(pkg/'models'/'hybrid_config.json', 'w'), indent=2)

lat_rows = []
for v in hybrid_cfg['variants']:
    hm = HybridFallModel(pkg/'models', v, num_threads=4)
    for i in range(50): hm.predict(X_test[i])            # khởi động
    ts = []
    for i in range(50, 50+1000):
        hm.predict(X_test[i % len(X_test)]); ts.append(hm.last_timing)
    t = pd.DataFrame(ts)
    lat_rows.append({'Phiên bản': f'TFLite {v}', 'Độ trễ TB (ms)': t.total_ms.mean(),
                     'Độ lệch chuẩn (ms)': t.total_ms.std(), 'Độ trễ P95 (ms)': t.total_ms.quantile(0.95),
                     'Trích đặc trưng (ms)': t.features_ms.mean(), 'Nhánh Transformer (ms)': t.transformer_branch_ms.mean(),
                     'Nhánh PMG (ms)': t.pmg_branch_ms.mean()})
lat_df = pd.DataFrame(lat_rows)
display(lat_df.round(2))
lat_df.to_csv(OUT/'latency_reference_cpu.csv', index=False)
print('Đo trên:', platform.processor() or platform.machine(), '— chỉ để tham khảo, không đưa vào bảng thiết bị.')


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use 

,Phiên bản,Độ trễ TB (ms),Độ lệch chuẩn (ms),Độ trễ P95 (ms),Trích đặc trưng (ms),Nhánh Transformer (ms),Nhánh PMG (ms)
0,TFLite float32,8.96,3.43,9.74,1.60,2.20,5.02
1,TFLite float16,8.87,2.78,9.87,1.60,2.21,4.92
2,TFLite dynamic_range,9.12,2.95,9.85,1.61,2.35,5.02
3,TFLite int8_full,16.58,2.34,18.60,1.57,4.94,9.95


Đo trên: x86_64 — chỉ để tham khảo, không đưa vào bảng thiết bị.


In [56]:
# ============================================================
# 8. ĐÓNG GÓI CHO THIẾT BỊ: edge_package.zip
#    Chép zip này sang bo mạch/laptop rồi mở notebook 04.
# ============================================================
for f in ['features_np.py', 'alert_manager.py', 'hybrid_tflite.py']:
    shutil.copy(f, pkg/f)
# 300 cửa sổ kiểm thử (có Fall) + xác suất Keras để kiểm tra trên thiết bị
sel = np.concatenate([np.flatnonzero(y_test == 1), np.flatnonzero(y_test == 0)[:200]])
np.savez_compressed(pkg/'parity_test_windows.npz', X=X_test[sel], y=y_test[sel], p_keras=p_keras[sel],
                    **{f'p_{v}': VARIANT_PROBS[v][sel] for v in VARIANT_PROBS})
# tham số bộ quản lý cảnh báo đã chọn ở notebook 02 (nếu có)
p3 = Path('/kaggle/working/P3_outputs/alert_params_chosen.json')
if p3.exists():
    shutil.copy(p3, pkg/'alert_params_chosen.json')
else:
    json.dump({'TAU': TAU, 'THETA_STAR': 0.0, 'T_POST': 3.0, 'T_COOL': 30.0, 'FPS': 18.0},
              open(pkg/'alert_params_chosen.json', 'w'), indent=2)
    print('Chưa có kết quả notebook 02 → tạm dùng θ_lie = 0°, FPS = 18. Cập nhật lại sau.')
print('Tải về:', shutil.make_archive('/kaggle/working/edge_package', 'zip', pkg))


Tải về: /kaggle/working/edge_package.zip
